# Random Forests: Bagging and Feature Randomness

A random forest reduces the instability of a decision tree by combining many trees. Every tree receives a bootstrap sample, and every split sees only a random subset of features.

## Learning goals

- Describe bootstrap sampling and random feature selection.
- Interpret `n_estimators`, `max_depth`, and `max_features`.
- Use predicted probabilities and impurity-based feature importances.

In [ ]:
from pathlib import Path
import sys

import numpy as np

repo_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src" / "rice_ml").is_dir()
)
sys.path.insert(0, str(repo_root / "src"))

from rice_ml.supervised_ml import RandomForestClassifier

In [ ]:
rng = np.random.default_rng(21)
X = rng.normal(size=(300, 5))
signal = 1.8 * X[:, 0] - 1.2 * X[:, 2] + 0.5 * X[:, 0] * X[:, 2]
y = np.where(signal + rng.normal(scale=0.6, size=300) > 0, "yes", "no")

indices = rng.permutation(len(X))
train, test = indices[:210], indices[210:]
X_train, X_test = X[train], X[test]
y_train, y_test = y[train], y[test]

In [ ]:
forest = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    max_features="sqrt",
    random_state=21,
)
forest.fit(X_train, y_train)

predictions = forest.predict(X_test)
probabilities = forest.predict_proba(X_test)
print(f"test accuracy: {np.mean(predictions == y_test):.3f}")
print("class order:", forest.classes_)
print("first five probability rows:\n", np.round(probabilities[:5], 3))

In [ ]:
ranked_features = sorted(
    enumerate(forest.feature_importances_),
    key=lambda pair: pair[1],
    reverse=True,
)
print("Feature importance ranking")
for feature, importance in ranked_features:
    print(f"feature {feature}: {importance:.3f}")

The data-generating process used features 0 and 2. A useful importance plot should reflect that signal. Remember that impurity-based importance describes how the fitted forest used a feature; it does not prove causality.

## Experiment

Fit forests with 1, 10, 50, and 150 trees. Record test accuracy. Then repeat with `max_features=None`. Explain how tree diversity and computation change.